# DriftWise: final figures and tables
Reads the result files saved in `MyDrive/DriftWise/results/` and writes paper-ready figures (PNG and PDF) to `results/plots/` and tables (CSV and LaTeX) to `results/tables/`.

**How to run:** no install or restart needed. Run the cells in order (cell 1 first; it asks for Google Drive access). If a result file is missing, that figure is skipped with a message and the rest still run.

| Figure | What it shows |
|---|---|
| 1 | Synthetic drift: a frozen model's overall error vs attack recall |
| 2 | UNSW-NB15: a frozen model's recall over time |
| 3 | Synthetic: all methods compared (with ARF) |
| 4 | Cost: updates, models trained, validation labels |
| 5 | Safety vs accuracy for every gate setting |
| 6 | UNSW-NB15: methods compared |
| 7 | UNSW-NB15: sequential gate vs simple margin rules |
| 8 | UNSW-NB15: what each accepted update really did |
| 9 | Gradual drift and delayed labels |

## 1. Setup

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/DriftWise"
RES, PLOTS, TABLES = f"{BASE}/results", f"{BASE}/results/plots", f"{BASE}/results/tables"
for d in (PLOTS, TABLES):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})
BLUE, LBLUE, GREY, RED, ORANGE = "#1f77b4", "#9ecae1", "#8c8c8c", "#d62728", "#ff7f0e"

def load(name):
    p = f"{RES}/{name}"
    if not os.path.exists(p):
        print(f"  [skipped] {name} not found in results/")
        return None
    return pd.read_csv(p)

def mean_ci(x):
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return float(x.mean()), 0.0
    return float(x.mean()), float(stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

def save_fig(fig, name):
    fig.savefig(f"{PLOTS}/{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"{PLOTS}/{name}.pdf", bbox_inches="tight")
    print("  saved figure:", name)

def save_table(df, name):
    df.to_csv(f"{TABLES}/{name}.csv")
    try:
        df.to_latex(f"{TABLES}/{name}.tex", float_format="%.3f")
    except Exception:
        pass
    print("  saved table:", name)

def barh_ci(ax, df, metric, order, colors, labels, title, xlim=None):
    """Horizontal bars: mean over seeds with a 95% confidence interval."""
    ms, hs = zip(*[mean_ci(df.loc[df.method == m, metric]) for m in order])
    y = np.arange(len(order))
    ax.barh(y, ms, xerr=hs, color=[colors[m] for m in order], capsize=3, alpha=0.95)
    ax.set_yticks(y)
    ax.set_yticklabels([labels[m] for m in order])
    ax.set_ylim(len(order) - 0.5, -0.5)       # first item on top (same on every shared axis)
    ax.set_title(title, fontsize=10)
    if xlim:
        ax.set_xlim(*xlim)
    ax.grid(axis="y", visible=False)
print("Setup done. Figures go to", PLOTS, "and tables to", TABLES)

## 2. Figure 1: synthetic drift (computed here)

In [ ]:
# ---------- Figure 1: drift on the synthetic stream, and why overall error is misleading ----------
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

def make_stream(seed, n=10000, drift_point=5000):
    r = np.random.default_rng(seed)
    p = np.where(np.arange(n) < drift_point, 0.05, 0.20)
    y = (r.random(n) < p).astype(int)
    X = r.normal(size=(n, 2))
    X[:drift_point, 0] += 2.0 * y[:drift_point]
    X[drift_point:, 1] += 2.0 * y[drift_point:]
    return X, y

def frozen_preds(X, y, start=4000):
    sc = StandardScaler().fit(X[:start])
    m = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m.fit(sc.transform(X[:start]), y[:start])
    return m.predict(sc.transform(X))

edges = np.arange(4000, 10001, 250)
err, rec1, rec0 = [], [], []
for seed in range(10, 30):
    X, y = make_stream(seed)
    p = frozen_preds(X, y)
    e_, a_, n_ = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        yy, pp = y[lo:hi], p[lo:hi]
        e_.append((yy != pp).mean())
        a_.append((pp[yy == 1] == 1).mean() if (yy == 1).any() else np.nan)
        n_.append((pp[yy == 0] == 0).mean())
    err.append(e_); rec1.append(a_); rec0.append(n_)
mid = (edges[:-1] + edges[1:]) / 2
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
for ax, data, title, col in [(axes[0], err, "Overall error rate", GREY),
                             (axes[1], rec1, "Attack recall (share of attacks caught)", RED)]:
    d = np.array(data, dtype=float)
    ax.plot(mid, np.nanmean(d, 0), color=col, lw=2)
    ax.fill_between(mid, np.nanmean(d, 0) - np.nanstd(d, 0), np.nanmean(d, 0) + np.nanstd(d, 0), color=col, alpha=0.2)
    ax.axvline(5000, color="k", ls="--", lw=1)
    ax.set_title(title)
    ax.set_xlabel("Record number")
axes[0].set_ylim(0, 0.5)
axes[1].set_ylim(0, 1.05)
axes[0].text(5050, 0.45, "drift", fontsize=9)
fig.suptitle("A frozen model after drift (synthetic stream, mean of 20 seeds, band = 1 sd)", y=1.03)
plt.tight_layout()
save_fig(fig, "fig1_synthetic_drift_frozen_model")
plt.show()

## 3. Figure 2: UNSW-NB15 frozen model (computed here from the dataset)

In [ ]:
# ---------- Figure 2: a frozen model on the real UNSW-NB15 data ----------
from sklearn.metrics import recall_score
path = f"{BASE}/data/processed/UNSW_NB15_DriftWise_clean_chronological.csv.gz"
if not os.path.exists(path):
    print("  [skipped] the UNSW-NB15 file is not in data/processed/")
else:
    df = pd.read_csv(path, low_memory=False)
    for c in ["sport", "dsport"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    drop_cols = ["Label", "attack_cat", "srcip", "dstip", "Stime", "Ltime", "Stime_UTC", "Ltime_UTC"]
    cat_cols = ["proto", "state", "service"]
    num_cols = [c for c in df.columns if c not in drop_cols + cat_cols]
    train_end = 17344
    keep = {c: set(df.iloc[:train_end][c].value_counts().head(10 if c == "proto" else 50).index) for c in cat_cols}

    def make_features(d):
        num = d[num_cols].fillna(-1).astype(float)
        num = np.sign(num) * np.log1p(np.abs(num))
        cats = d[cat_cols].copy()
        for c in cat_cols:
            cats[c] = cats[c].where(cats[c].isin(keep[c]), "other")
        return pd.concat([num, pd.get_dummies(cats, dtype=float)], axis=1)

    last_attack = int(np.where(df["Label"].values == 1)[0].max())
    N = last_attack + 1
    columns = make_features(df.iloc[:train_end]).columns
    X_raw = make_features(df.iloc[:N]).reindex(columns=columns, fill_value=0.0).to_numpy()
    y_all = df["Label"].to_numpy()[:N]
    sc = StandardScaler().fit(X_raw[:train_end])
    mdl = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    mdl.fit(sc.transform(X_raw[:train_end]), y_all[:train_end])
    pred = mdl.predict(sc.transform(X_raw))

    starts = list(range(train_end, N, 5000))
    rows = []
    for s in starts:
        e = min(s + 5000, N)
        yy, pp = y_all[s:e], pred[s:e]
        if yy.sum() < 30:
            continue
        rows.append({"mid": (s + e) / 2, "attack_rate": 100 * yy.mean(),
                     "attack_recall": (pp[yy == 1] == 1).mean(), "normal_recall": (pp[yy == 0] == 0).mean()})
    b = pd.DataFrame(rows)
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
    axes[0].bar(b["mid"], b["attack_rate"], width=4200, color=GREY)
    axes[0].set_ylabel("Attacks in block (%)")
    axes[1].plot(b["mid"], b["attack_recall"], "o-", color=RED, label="attack recall")
    axes[1].plot(b["mid"], b["normal_recall"], "s-", color=BLUE, label="normal recall")
    axes[1].set_ylim(0.7, 1.01)
    axes[1].set_ylabel("Recall")
    axes[1].set_xlabel("Row number (chronological order)")
    for ax in axes:
        ax.axvspan(117000, N, color=ORANGE, alpha=0.12)
    axes[1].text(118000, 0.72, "later period: attack recall falls", fontsize=9)
    axes[1].legend(loc="lower left")
    fig.suptitle("UNSW-NB15: a model trained once on the first 17,344 rows, scored in blocks of 5,000 rows", y=0.99)
    plt.tight_layout()
    save_fig(fig, "fig2_real_data_frozen_model")
    plt.show()

## 4. Figure 3 and Table 1: synthetic main comparison

In [ ]:
# ---------- Figure 3 + Table 1: synthetic main comparison (pattern drift, 30 fresh seeds) ----------
sched = load("ega_v2_vs_schedules_pattern_lag0.csv")
fresh = load("ega_v2_fresh_seeds_pattern_and_rate_only.csv")
single = load("single_delayed_retrain_pattern_lag0.csv")
arf = load("arf_baseline_comparison.csv")

parts = []
def never_update_rows():
    rows = []
    for seed in range(10, 40):
        X, y = make_stream(seed)
        p = frozen_preds(X, y)
        ye, pe = y[5000:], p[5000:]
        r0, r1 = (pe[ye == 0] == 0).mean(), (pe[ye == 1] == 1).mean()
        rows.append({"method": "never update", "seed": seed, "balanced_acc": (r0 + r1) / 2,
                     "worst_class_recall": min(r0, r1), "n_updates": 0})
    return pd.DataFrame(rows)
parts.append(never_update_rows())
if sched is not None:
    parts.append(sched[["method", "seed", "balanced_acc", "worst_class_recall", "n_updates"]])
if fresh is not None:
    f = fresh[(fresh.condition == "pattern_drift") & (fresh.method == "ega (stop at first accept)")].copy()
    f["method"] = "EGA v1: stop at first accept"
    parts.append(f[["method", "seed", "balanced_acc", "worst_class_recall", "n_updates"]])
if single is not None:
    parts.append(single[["method", "seed", "balanced_acc", "worst_class_recall", "n_updates"]])
if arf is not None:
    a = arf[arf.method.str.startswith("ARF")].copy()
    a["method"] = a["method"].str.replace("ARF", "ARF (10 seeds)", regex=False)
    a["n_updates"] = np.nan
    parts.append(a[["method", "seed", "balanced_acc", "worst_class_recall", "n_updates"]])
S = pd.concat(parts, ignore_index=True)

LAB = {"never update": "Never update",
       "every_alarm (retrain once)": "Retrain once at the alarm",
       "1 retrain at alarm+300 (300 labels)": "Retrain once, 300 later",
       "1 retrain at alarm+600 (300 labels)": "Retrain once, 600 later",
       "1 retrain at alarm+600 (600 labels)": "Retrain once, 600 later (600 labels)",
       "retrain at alarm and +500 (no gate)": "Retrain at alarm and +500",
       "retrain at alarm, +300, +600 (no gate)": "Retrain at alarm, +300, +600",
       "EGA v1: stop at first accept": "Gated, stop at first accept (v1)",
       "ega_v2 (gated, keep improving)": "Gated, keep improving (v2)",
       "ARF (10 seeds) (default)": "ARF default (10 seeds)",
       "ARF (10 seeds) (attacks shown 3x)": "ARF, attacks shown 3x (10 seeds)"}
order = [m for m in LAB if m in set(S.method)]
colors = {m: GREY for m in order}
for m in order:
    if "v2" in LAB[m]: colors[m] = BLUE
    elif "v1" in LAB[m]: colors[m] = LBLUE
    elif "ARF" in LAB[m]: colors[m] = RED
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
barh_ci(axes[0], S, "balanced_acc", order, colors, LAB, "Balanced accuracy after the drift", xlim=(0.45, 0.85))
barh_ci(axes[1], S, "worst_class_recall", order, colors, LAB, "Worst-class recall after the drift", xlim=(0.0, 0.85))
fig.suptitle("Synthetic pattern drift (mean over fresh seeds, bars = 95% CI)", y=1.0)
plt.tight_layout()
save_fig(fig, "fig3_synthetic_main_comparison")
plt.show()

rows = []
for m in order:
    d = S[S.method == m]
    ba, bh = mean_ci(d.balanced_acc); wr, wh = mean_ci(d.worst_class_recall)
    rows.append({"method": LAB[m], "seeds": d.seed.nunique(), "balanced acc": ba, "+/- 95% CI": bh,
                 "worst-class recall": wr, "+/- CI": wh, "updates": d.n_updates.mean()})
T1 = pd.DataFrame(rows).set_index("method")
save_table(T1, "table1_synthetic_main_comparison")
display(T1.round(3))

## 5. Figures 4 and 5, Tables 2 and 3: cost and gate trade-off

In [ ]:
# ---------- Figure 4 + Table 2: what each method costs (synthetic, 20 seeds) ----------
cost = load("ega_v2_cost_comparison.csv")
if cost is not None:
    LABC = {"every_alarm (retrain once)": "Retrain once at the alarm",
            "1 retrain at alarm+300": "Retrain once, 300 later",
            "retrain at alarm, +300, +600": "Retrain at alarm, +300, +600",
            "ega_v2 (gated)": "Gated, keep improving (v2)"}
    order = [m for m in LABC if m in set(cost.method)]
    cols_ = {m: (BLUE if "v2" in LABC[m] else GREY) for m in order}
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for ax, metric, title, log in [(axes[0], "updates", "Models deployed (updates)", False),
                                   (axes[1], "models_trained", "Models trained in total", True),
                                   (axes[2], "validation_labels_used", "Validation labels used", True)]:
        barh_ci(ax, cost, metric, order, cols_, LABC, title)
        if log:
            ax.set_xscale("symlog", linthresh=1)
    fig.suptitle("Cost of each method: pattern drift, mean of 20 seeds", y=1.02)
    plt.tight_layout()
    save_fig(fig, "fig4_cost_comparison")
    plt.show()
    T2 = cost.groupby("method")[["balanced_acc", "updates", "models_trained", "gate_tests", "validation_labels_used", "seconds"]].mean().loc[order]
    T2.index = [LABC[m] for m in T2.index]
    save_table(T2, "table2_cost")
    display(T2.round(3))

# ---------- Figure 5 + Table 3: safety vs accuracy for every gate setting ----------
tr = load("gate_tradeoff_summary.csv")
if tr is not None:
    tr = tr.set_index(tr.columns[0])
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for prefix, color, label in [("sequential", BLUE, "Sequential gate (confidence level alpha)"),
                                 ("point", ORANGE, "Simple rule (safety margin)")]:
        d = tr[[i.startswith(prefix) for i in tr.index]]
        ax.plot(d["harmful accepts /1000 checks"], d["balanced_acc"], "o-", color=color, label=label)
        for name, r in d.iterrows():
            ax.annotate(name.split(", ")[1], (r["harmful accepts /1000 checks"], r["balanced_acc"]),
                        textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlabel("Harmful accepts per 1,000 checks on streams with no drift (lower = safer)")
    ax.set_ylabel("Balanced accuracy under drift (higher = better)")
    ax.set_title("Safety vs accuracy: sequential gate and simple margin rule")
    ax.legend()
    plt.tight_layout()
    save_fig(fig, "fig5_gate_tradeoff")
    plt.show()
    save_table(tr, "table3_gate_tradeoff")
    display(tr.round(3))

## 6. Figures 6 to 8, Table 4: real data

In [ ]:
# ---------- Figures 6-8 + Table 4: real data (UNSW-NB15) ----------
def barh_dots(ax, df, metric, order, colors, labels, title, xlim=None):
    """Bars = mean over seeds; dots = individual seeds (only a few seeds, so no confidence interval)."""
    y = np.arange(len(order))
    ax.barh(y, [df.loc[df.method == m, metric].mean() for m in order],
            color=[colors[m] for m in order], alpha=0.9)
    for i, m in enumerate(order):
        v = df.loc[df.method == m, metric].to_numpy()
        ax.scatter(v, np.full(len(v), i), color="k", s=14, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels([labels[m] for m in order]); ax.set_ylim(len(order) - 0.5, -0.5)
    ax.set_title(title, fontsize=10)
    if xlim: ax.set_xlim(*xlim)
    ax.grid(axis="y", visible=False)

stab = load("real_data_stability.csv")
if stab is not None:
    LABR = {"never update": "Never update",
            "periodic (every 10,000, newest 50,000)": "Periodic retrain (every 10,000 rows)",
            "retrain at every UP alarm (newest 20,000)": "Retrain at each alarm (20,000 rows)",
            "retrain at every UP alarm (newest 30,000)": "Retrain at each alarm (30,000 rows)",
            "EGA v2, keep improving": "Gated, keep improving (v2)"}
    order = [m for m in LABR if m in set(stab.method)]
    colr = {m: (BLUE if "v2" in LABR[m] else GREY) for m in order}
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    barh_dots(axes[0], stab, "ALL balanced_acc", order, colr, LABR, "Balanced accuracy, whole stream", xlim=(0.94, 0.985))
    barh_dots(axes[1], stab, "DRIFT balanced_acc", order, colr, LABR, "Balanced accuracy, later period", xlim=(0.92, 0.98))
    barh_dots(axes[2], stab, "DRIFT attack_recall", order, colr, LABR, "Attack recall, later period", xlim=(0.85, 0.98))
    fig.suptitle("UNSW-NB15: methods compared (bars = mean of 3 random seeds, dots = seeds)", y=1.02)
    plt.tight_layout()
    save_fig(fig, "fig6_real_data_methods")
    plt.show()

gab = load("real_data_gate_ablation.csv")
if gab is not None:
    LABG = {"never update": "Never update",
            "periodic (every 10,000, newest 50,000)": "Periodic retrain",
            "EGA v2: sequential gate (alpha=0.05)": "Gated: sequential gate",
            "EGA v2: point rule, margin=0": "Gated: simple rule, margin 0",
            "EGA v2: point rule, margin=0.01": "Gated: simple rule, margin 0.01",
            "EGA v2: point rule, margin=0.02": "Gated: simple rule, margin 0.02"}
    order = [m for m in LABG if m in set(gab.method)]
    colg = {m: (BLUE if "sequential" in m else ORANGE if "point" in m else GREY) for m in order}
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
    barh_dots(axes[0], gab, "DRIFT bal_acc", order, colg, LABG, "Balanced accuracy, later period", xlim=(0.92, 0.98))
    barh_dots(axes[1], gab, "DRIFT attack_recall", order, colg, LABG, "Attack recall, later period", xlim=(0.85, 0.98))
    barh_dots(axes[2], gab, "updates", order, colg, LABG, "Models deployed")
    barh_dots(axes[3], gab, "models trained", order, colg, LABG, "Models trained in total")
    axes[3].set_xscale("log")
    fig.suptitle("UNSW-NB15: sequential gate vs simple margin rules (2 random seeds)", y=1.02)
    plt.tight_layout()
    save_fig(fig, "fig7_real_data_gate_ablation")
    plt.show()

eff = load("real_data_gate_ablation_update_effects.csv")
if eff is not None:
    order = [m for m in eff.method.unique()]
    fig, ax = plt.subplots(figsize=(8.5, 4))
    rng_ = np.random.default_rng(0)
    for i, m in enumerate(order):
        v = eff.loc[eff.method == m, "realized_change"].to_numpy()
        ax.scatter(v, i + rng_.uniform(-0.12, 0.12, len(v)), color=BLUE if "sequential" in m else ORANGE, s=22, alpha=0.8)
        ax.scatter([v.mean()], [i], marker="D", color="k", s=40, zorder=3)
    ax.axvline(0, color="k", lw=1)
    ax.axvspan(-0.01, 0.01, color=GREY, alpha=0.2)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([m.replace("EGA v2: ", "") for m in order])
    ax.invert_yaxis()
    ax.set_xlabel("Change in balanced accuracy over the next 10,000 rows (new model minus old); diamond = mean")
    ax.set_title("What each accepted update really did (grey band = within +/-0.01)", fontsize=10)
    ax.grid(axis="y", visible=False)
    plt.tight_layout()
    save_fig(fig, "fig8_real_data_update_effects")
    plt.show()

if gab is not None:
    T4 = gab.groupby("method")[["updates", "models trained", "ALL bal_acc", "DRIFT bal_acc", "DRIFT attack_recall"]].mean().loc[[m for m in LABG if m in set(gab.method)]]
    if eff is not None:
        g = eff.groupby("method")["realized_change"]
        T4["updates judged in hindsight"] = g.count()
        T4["mean real change"] = g.mean()
        T4["harmful (< -0.01)"] = g.apply(lambda s: int((s < -0.01).sum()))
        T4["helpful (> +0.01)"] = g.apply(lambda s: int((s > 0.01).sum()))
    T4.index = [LABG[m] for m in T4.index]
    save_table(T4, "table4_real_data_gate_ablation")
    display(T4.round(3))

## 7. Figure 9: gradual drift and delayed labels

In [ ]:
# ---------- Figure 9: gradual drift and delayed labels ----------
gd = load("ega_v2_vs_baselines_gradual_and_delayed.csv")
if gd is not None:
    LABD = {"every_alarm (retrain once)": "Retrain once at the alarm",
            "retrain at alarm, +300, +600 (no gate)": "Retrain at alarm, +300, +600",
            "ega_v2 (gated)": "Gated, keep improving (v2)"}
    gd = gd[gd.method.isin(LABD)].copy()
    gd["setting"] = gd["condition"].str.replace("_", " ") + ", label delay " + gd["lag"].astype(int).astype(str)
    settings = list(dict.fromkeys(gd["setting"]))
    methods = [m for m in LABD if m in set(gd.method)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    width = 0.8 / len(methods)
    for ax, metric, title in [(axes[0], "balanced_acc", "Balanced accuracy"), (axes[1], "n_updates", "Models deployed (updates)")]:
        for j, m in enumerate(methods):
            vals = [mean_ci(gd[(gd.setting == s) & (gd.method == m)][metric]) for s in settings]
            ax.bar(np.arange(len(settings)) + j * width, [v[0] for v in vals], width, yerr=[v[1] for v in vals],
                   capsize=2, color=BLUE if "v2" in LABD[m] else (GREY if "alarm," not in LABD[m] else "#c7c7c7"), label=LABD[m])
        ax.set_xticks(np.arange(len(settings)) + width * (len(methods) - 1) / 2)
        ax.set_xticklabels(settings, rotation=15, ha="right", fontsize=8)
        ax.set_title(title)
        ax.grid(axis="x", visible=False)
    axes[0].set_ylim(0.6, 0.85)
    axes[0].legend(fontsize=8)
    fig.suptitle("Gradual drift and delayed labels (mean of 20 seeds, bars = 95% CI)", y=1.02)
    plt.tight_layout()
    save_fig(fig, "fig9_gradual_and_delayed")
    plt.show()

print("\nFiles now in results/plots:", sorted(os.listdir(PLOTS)))
print("Files now in results/tables:", sorted(os.listdir(TABLES)))